In [1]:
import torch
import CIFAR10.config as c
from utils import trainer as t
from utils import trainer_dfa as t_dfa
import random
import numpy as np
from utils.layers.linear_rfa import LinearRFA
from utils.layers.linear_dfa import LinearDFA

In [2]:
def networks_have_same_state(net1, net2):
    """Check if two networks have the same initial weight state.
    Handles standard Linear, LinearRFA, and LinearDFA layers.
    """
    net1_layers = [m for m in net1.modules() if isinstance(m, (torch.nn.Linear, LinearRFA, LinearDFA))]
    net2_layers = [m for m in net2.modules() if isinstance(m, (torch.nn.Linear, LinearRFA, LinearDFA))]

    if len(net1_layers) != len(net2_layers):
        print(f"Layer count mismatch: {len(net1_layers)} vs {len(net2_layers)}")
        return False

    for i, (layer1, layer2) in enumerate(zip(net1_layers, net2_layers)):
        if not torch.equal(layer1.weight.data, layer2.weight.data):
            print(f"  Weight mismatch in layer {i}")
            return False

        if layer1.bias is not None and layer2.bias is not None:
            if not torch.equal(layer1.bias.data, layer2.bias.data):
                print(f"  Bias mismatch in layer {i}")
                return False

    return True

In [ ]:
import random
import torch

activation_pool = [
    torch.nn.ReLU(),
    torch.nn.GELU(),
    torch.nn.Tanh(),
    torch.nn.Sigmoid(),
    torch.nn.SiLU(),
    torch.nn.LeakyReLU(0.01),
    torch.nn.ELU(),
    torch.nn.Softplus(),
]

curr_activation = activation_pool[2]
num_experiments = 1
#random_seeds = [int(random.uniform(100, 100000)) for _ in range(num_experiments)]
random_seeds = [4764]
uid = random.randint(1000, 9999)
print(f"Generated {len(random_seeds)} unique seeds: {random_seeds[:5]}... (showing first 5)")
hidden_layers = [1024,512,256,128]
#init_methods = ["arora_balanced", "kaiming", "he", "glorot", "orthogonal", "xavier"]
init_methods = ["xavier"]

for exp_num, random_seed in enumerate(random_seeds):
    for i, init_method in enumerate(init_methods):
        print(f"\n{'='*60}")
        print(f"Experiment {exp_num + 1}/{num_experiments} - Seed: {random_seed} - Activation: {curr_activation.__class__.__name__}")
        print(f"{'='*60}")

        config_standard = c.get_config(
            hidden_layers=hidden_layers,
            activation=curr_activation,
            run_name = f"bp_{random_seed}_{curr_activation.__class__.__name__.lower()}_{init_method}",
            run_id=f"bp_{random_seed}_{uid}_{init_method}",
            SEED=random_seed,
            mode="BP",
            init_method= init_method,
        )

        config_rfa = c.get_config(
            hidden_layers=hidden_layers,
            activation=curr_activation,
            run_name = f"rfa_{random_seed}_{curr_activation.__class__.__name__.lower()}_{init_method}",
            run_id=f"rfa_{random_seed}_{uid}_{init_method}",
            SEED=random_seed,
            mode="RFA",
            init_method= init_method, 
        )

        config_dfa = c.get_config(
            hidden_layers=hidden_layers,
            activation=curr_activation,
            run_name = f"dfa_{random_seed}_{curr_activation.__class__.__name__.lower()}_{init_method}",
            run_id=f"dfa_{random_seed}_{uid}_{init_method}",
            SEED=random_seed,
            mode="DFA",
            init_method= init_method
        )
        '''
        
        print("Checking if standard, RFA, and DFA start from the same initial state...")
        if not networks_have_same_state(config_standard['net'], config_rfa['net']):
            print("ERROR: Standard and RFA are NOT in the same state!")
            continue
        if not networks_have_same_state(config_standard['net'], config_dfa['net']):
            print("ERROR: Standard and DFA are NOT in the same state!")
            continue
            '''
        
        print("All three networks start from the SAME initial state!")
        print("\nTraining standard model...")
        #t.train_network(config_standard, num_epochs=100, checkpoint_interval=301)
        print("\nTraining RFA model...")
        t.train_network(config_rfa, num_epochs=100, checkpoint_interval=301)
        print("\nTraining DFA model with the global-output-error path...")
        #t_dfa.train_network(config_dfa, num_epochs=300, checkpoint_interval=301)
        print(f"✓ Experiment {exp_num + 1}, Activation: {curr_activation.__class__.__name__} completed successfully!")

print(f"\n{'='*60}")
print(f"All {num_experiments} experiments completed")
print(f"{'='*60}")

Generated 1 unique seeds: [4764]... (showing first 5)

Experiment 1/1 - Seed: 4764 - Activation: Tanh
[Linear(in_features=3072, out_features=1024, bias=False), Linear(in_features=1024, out_features=512, bias=False), Linear(in_features=512, out_features=256, bias=False), Linear(in_features=256, out_features=128, bias=False), Linear(in_features=128, out_features=10, bias=False)]
Initializing layer Linear(in_features=3072, out_features=1024, bias=False) with method 'xavier'
Layer shape: torch.Size([1024, 3072]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=1024, out_features=512, bias=False) with method 'xavier'
Layer shape: torch.Size([512, 1024]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=512, out_features=256, bias=False) with method 'xavier'
Layer shape: torch.Size([256, 512]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=256, out_features=128, bias=False) with method '

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Initializing Wandb:


wandb: Currently logged in as: g-aravindadithya (ICLR_2027) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


EPOCH:  1
Time:  7.317486524581909
EPOCH:  2
Time:  5.887712478637695
EPOCH:  3
Time:  5.321606397628784
EPOCH:  4
Time:  4.829303026199341
EPOCH:  5
Time:  5.740325450897217
EPOCH:  6
Time:  5.048269271850586
EPOCH:  7
Time:  5.222754001617432
EPOCH:  8
Time:  5.000157833099365
EPOCH:  9
Time:  4.779397487640381
EPOCH:  10
Time:  6.731898784637451
EPOCH:  11
Time:  5.348603010177612
EPOCH:  12
Time:  5.187368869781494
EPOCH:  13
Time:  5.257943391799927
EPOCH:  14
Time:  4.905493259429932
EPOCH:  15
Time:  4.896942377090454
EPOCH:  16
Time:  5.344099044799805
EPOCH:  17
Time:  5.177020788192749
EPOCH:  18
Time:  5.149727821350098
EPOCH:  19
Time:  5.1409077644348145
EPOCH:  20
Time:  6.226947069168091
EPOCH:  21
Time:  5.57960319519043
EPOCH:  22
Time:  5.076263904571533
EPOCH:  23
Time:  5.072575807571411
EPOCH:  24
Time:  5.03093957901001
EPOCH:  25
Time:  4.979042053222656
EPOCH:  26
Time:  5.505112171173096
EPOCH:  27
Time:  7.664355993270874
EPOCH:  28
Time:  6.778583526611328
EP

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x772f87439340>> (for post_run_cell), with arguments args (<ExecutionResult object at 772f87361b50, execution_count=3 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 772f873619a0, raw_cell="import random
import torch

activation_pool = [
  .." transformed_cell="import random
import torch

activation_pool = [
  .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://dev-container%2B7b227265706f7369746f727950617468223a2268747470733a2f2f6769746875622e636f6d2f61726176696e64616469746879612f4d61747269672e6769742f747265652f6d6173746572222c22766f6c756d654e616d65223a224d61747269672d6d61737465722d62353132393936373030653663336536623532643661366636306537636564613937326161383937663434346361666233353766633066363632323230663739222c22666f6c646572223a224d6174726967222c2273657474696e6773223a7b22636f6e74657874223a226465736b746f702d6c696e7578227d2c2

ConnectionResetError: Connection lost

: 